# TAPT: task-adaptive pretraining for XLM-R-large (Google Colab, L4 GPU)

Two phases:
1. **MLM pretraining on the training texts only** — continue `xlm-roberta-large`'s masked-language-model objective for a few epochs on the 17,769 `"tag | feedback"` training texts, adapting its representations to informal Azerbaijani/Russian/English micro-mobility language before any label is seen.
2. **Standard fine-tuning CV** — the exact validated 5-epoch classification recipe, starting from the adapted checkpoint instead of the generic one, on the same 5 folds as every other model.

**Compliance note (deliberate design)**: this notebook never loads `test.csv`. The MLM corpus is `train.csv` text only — using test text in any training step (including unlabeled MLM) would violate the competition's prohibition on utilizing the test dataset for training.

**Adoption gate (pre-registered)**: the TAPT model replaces the plain XLM-R-large (CV 0.9207) only if its 5-fold CV mean is **>= 0.9240** (i.e. a gain clearly above the ~±0.003 noise floor). Below that: keep the plain model, close the TAPT question.

Runtime: **L4 GPU**. Expected cost: MLM ~20-30 min + CV ~1h ≈ 8-10 compute units.

## Setup

In [1]:
%pip install -q datasets scikit-learn

In [2]:
import math
import random

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

RANDOM_SEED = 42
BASE_MODEL = "xlm-roberta-large"
MAX_LENGTH = 96
N_SPLITS = 5
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 1e-5
WARMUP_FRACTION = 0.1

# MLM phase settings
MLM_EPOCHS = 10
MLM_PROBABILITY = 0.15

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
set_seed(RANDOM_SEED)

print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("bf16:", USE_BF16, "| fp16:", USE_FP16)

Device: NVIDIA L4
bf16: True | fp16: False


In [3]:
import platform

print("torch:", torch.__version__)
import transformers
print("transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Python:", platform.python_version())

torch: 2.11.0+cu128
transformers: 5.12.1
CUDA: 12.8
GPU: NVIDIA L4
Python: 3.12.13


## Mount Drive and load data (train.csv ONLY — see compliance note above)

In [4]:
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/apar_data"
OUTPUT_DIR = "/content/drive/MyDrive/apar_xlmrlarge_tapt"
TAPT_DIR = f"{OUTPUT_DIR}/tapt_checkpoint"  # adapted encoder after MLM phase

Mounted at /content/drive


In [5]:
import re

TAG_MISSING_VALUE = "missing"


def clean_text(text: str) -> str:
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["feedback"] = df["feedback"].astype(str).apply(clean_text)
    df["tag"] = df["tag"].fillna(TAG_MISSING_VALUE)
    df["input_text"] = df["tag"] + " | " + df["feedback"]
    return df


train_df = prepare_dataframe(pd.read_csv(f"{DATA_DIR}/train.csv"))

LABELS = ["technical_support", "customer_support", "other"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}
train_df["label_id"] = train_df["label"].map(label2id)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(train_df.shape)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

(17769, 6)


## Phase 1: MLM pretraining on train text

The same `"tag | feedback"` strings the classifier will later see, tokenized identically. `DataCollatorForLanguageModeling` masks 15% of tokens per batch; the model learns to reconstruct them. 10 epochs at lr 1e-5 — enough adaptation to the domain without memorizing 17.7k short texts with a 560M model. Labels are never used in this phase.

In [6]:
mlm_ds = Dataset.from_pandas(train_df[["input_text"]])
mlm_ds = mlm_ds.map(
    lambda b: tokenizer(b["input_text"], truncation=True, max_length=MAX_LENGTH),
    batched=True, remove_columns=["input_text"],
)

mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=MLM_PROBABILITY)

mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)
mlm_model = mlm_model.float()

mlm_steps_per_epoch = math.ceil(len(mlm_ds) / (BATCH_SIZE * GRAD_ACCUM_STEPS))
mlm_args = TrainingArguments(
    output_dir="/content/mlm_output",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=MLM_EPOCHS,
    warmup_steps=int(WARMUP_FRACTION * mlm_steps_per_epoch * MLM_EPOCHS),
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    bf16=USE_BF16,
    fp16=USE_FP16,
    logging_steps=100,
    report_to="none",
    seed=RANDOM_SEED,
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_ds,
    data_collator=mlm_collator,
)
mlm_trainer.train()

Map:   0%|          | 0/17769 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-large
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,9.928851
200,6.783383
300,5.910583
400,5.331497
500,5.012711
600,4.788521
700,4.516619
800,4.360923
900,4.379587
1000,4.263590


TrainOutput(global_step=5560, training_loss=3.7250331796330514, metrics={'train_runtime': 3206.4534, 'train_samples_per_second': 55.416, 'train_steps_per_second': 1.734, 'total_flos': 2.300466067363649e+16, 'train_loss': 3.7250331796330514, 'epoch': 10.0})

In [7]:
mlm_model.save_pretrained(TAPT_DIR)
tokenizer.save_pretrained(TAPT_DIR)
print(f"Saved TAPT-adapted checkpoint to {TAPT_DIR}")

del mlm_model, mlm_trainer
torch.cuda.empty_cache()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved TAPT-adapted checkpoint to /content/drive/MyDrive/apar_xlmrlarge_tapt/tapt_checkpoint


## Phase 2: standard fine-tuning CV from the adapted checkpoint

Same 5-fold classification recipe used throughout this project (5-fold `StratifiedKFold(random_state=42)`, weighted cross-entropy, batch 16 / grad-accum 2 / lr 1e-5 / bf16, 5 epochs), starting `from_pretrained(TAPT_DIR)` -- the MLM-adapted checkpoint -- instead of the generic pretrained one. The MISSING/UNEXPECTED head-swap warnings in the load report below are expected (MLM head dropped, classification head freshly initialized).

In [8]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def tokenize_batch(batch):
    return tokenizer(batch["input_text"], truncation=True, max_length=MAX_LENGTH)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


def build_training_args(output_dir: str, num_train_samples: int, num_train_epochs: int = 5) -> TrainingArguments:
    steps_per_epoch = math.ceil(num_train_samples / (BATCH_SIZE * GRAD_ACCUM_STEPS))
    warmup_steps = int(WARMUP_FRACTION * steps_per_epoch * num_train_epochs)
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        per_device_eval_batch_size=32,
        num_train_epochs=num_train_epochs,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        bf16=USE_BF16,
        fp16=USE_FP16,
        logging_steps=50,
        report_to="none",
        seed=RANDOM_SEED,
    )


def run_fold(fold_idx, train_idx, val_idx):
    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].reset_index(drop=True)

    class_weights = compute_class_weight(
        "balanced", classes=np.arange(len(LABELS)), y=fold_train_df["label_id"]
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    train_ds = Dataset.from_pandas(fold_train_df[["input_text", "label_id"]]).rename_column("label_id", "labels")
    val_ds = Dataset.from_pandas(fold_val_df[["input_text", "label_id"]]).rename_column("label_id", "labels")
    train_ds = train_ds.map(tokenize_batch, batched=True)
    val_ds = val_ds.map(tokenize_batch, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        TAPT_DIR, num_labels=len(LABELS), id2label=id2label, label2id=label2id
    )
    model = model.float()

    training_args = build_training_args(
        output_dir=f"/content/fold_{fold_idx}_output", num_train_samples=len(fold_train_df)
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )
    trainer.train()

    val_logits = trainer.predict(val_ds).predictions
    val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
    fold_macro_f1 = f1_score(fold_val_df["label_id"], val_probs.argmax(axis=1), average="macro")

    del model, trainer
    torch.cuda.empty_cache()

    return val_probs, fold_macro_f1

## Smoke test: fold 1 (mechanical validation)

Confirmed the adapted checkpoint loads and trains correctly before committing to the full 5-fold run. Reference point: a plain (non-TAPT) XLM-R-large fine-tune scored 0.9272 on this same fold in its own CV (0.9298 in its own fold-1 smoke test -- same config, ±0.003 run-to-run noise). This single fold was not used for the adoption decision, which is based on the full 5-fold mean below.

In [9]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
first_train_idx, first_val_idx = next(skf.split(train_df, train_df["label_id"]))

_, smoke_test_macro_f1 = run_fold(0, first_train_idx, first_val_idx)
print(f"Smoke test (fold 1 only) Macro F1: {smoke_test_macro_f1:.4f}")

Map:   0%|          | 0/14215 [00:00<?, ? examples/s]

Map:   0%|          | 0/3554 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/apar_xlmrlarge_tapt/tapt_checkpoint
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.533092,0.261521,0.914560
2,0.453330,0.223608,0.929864
3,0.265609,0.327700,0.923864
4,0.210976,0.347770,0.926850
5,0.166828,0.410219,0.928984


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Smoke test (fold 1 only) Macro F1: 0.9299


## Full 5-fold cross-validation

In [10]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

oof_probs = np.zeros((len(train_df), len(LABELS)))
fold_macro_f1_scores = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label_id"]), start=1):
    val_probs, fold_macro_f1 = run_fold(fold_idx, train_idx, val_idx)
    oof_probs[val_idx] = val_probs
    fold_macro_f1_scores.append(fold_macro_f1)
    print(f"Fold {fold_idx}: Macro F1 = {fold_macro_f1:.4f}")

print(
    f"\nMean Macro F1: {np.mean(fold_macro_f1_scores):.4f} "
    f"(+/- {np.std(fold_macro_f1_scores):.4f})"
)
print("Plain XLM-R-large reference: 0.9207 (+/- 0.0066). Adoption gate: mean >= 0.9240.")

Map:   0%|          | 0/14215 [00:00<?, ? examples/s]

Map:   0%|          | 0/3554 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/apar_xlmrlarge_tapt/tapt_checkpoint
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.522396,0.242982,0.906752
2,0.433856,0.211145,0.931606
3,0.248922,0.311712,0.927609
4,0.186384,0.337020,0.927367
5,0.174984,0.388586,0.934036


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 5: Macro F1 = 0.9172

Mean Macro F1: 0.9272 (+/- 0.0060)
Plain XLM-R-large reference: 0.9207 (+/- 0.0066). Adoption gate: mean >= 0.9240.


In [11]:
oof_preds = oof_probs.argmax(axis=1)
print(classification_report(train_df["label_id"], oof_preds, target_names=LABELS, digits=4))

                   precision    recall  f1-score   support

technical_support     0.9595    0.9646    0.9620     10642
 customer_support     0.8810    0.9032    0.8920      3327
            other     0.9454    0.9105    0.9276      3800

         accuracy                         0.9415     17769
        macro avg     0.9286    0.9261    0.9272     17769
     weighted avg     0.9418    0.9415    0.9416     17769



## Save out-of-fold predictions

In [12]:
oof_df = train_df[["id", "label"]].copy()
for i, label in enumerate(LABELS):
    oof_df[f"prob_{label}"] = oof_probs[:, i]
oof_df["oof_pred"] = [LABELS[i] for i in oof_preds]

oof_df.to_csv(f"{OUTPUT_DIR}/xlmrlarge_tapt_oof_predictions.csv", index=False)
print(f"Saved OOF predictions to {OUTPUT_DIR}/xlmrlarge_tapt_oof_predictions.csv")

Saved OOF predictions to /content/drive/MyDrive/apar_xlmrlarge_tapt/xlmrlarge_tapt_oof_predictions.csv


## Final model: retrain on 100% of the data

This retrain only runs once the pre-registered adoption gate (mean CV >= 0.9240) has passed. The 5-fold mean above (0.9272) cleared it, so the checkpoint below is the one retrained on 100% of the training data and deployed.

In [13]:
FINAL_NUM_EPOCHS = 5  # adjust from fold logs if the gate passed

final_class_weights = compute_class_weight(
    "balanced", classes=np.arange(len(LABELS)), y=train_df["label_id"]
)
final_class_weights = torch.tensor(final_class_weights, dtype=torch.float)

final_train_ds = Dataset.from_pandas(train_df[["input_text", "label_id"]]).rename_column("label_id", "labels")
final_train_ds = final_train_ds.map(tokenize_batch, batched=True)

final_model = AutoModelForSequenceClassification.from_pretrained(
    TAPT_DIR, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)
final_model = final_model.float()

final_training_args = build_training_args(
    output_dir="/content/final_model_output",
    num_train_samples=len(train_df),
    num_train_epochs=FINAL_NUM_EPOCHS,
)
final_training_args.eval_strategy = "no"
final_training_args.save_strategy = "no"
final_training_args.load_best_model_at_end = False

final_trainer = WeightedTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=final_train_ds,
    data_collator=data_collator,
    class_weights=final_class_weights,
)
final_trainer.train()

n_params = sum(p.numel() for p in final_model.parameters())
print(f"Parameter count: {n_params:,} ({n_params / 1e6:.0f}M)")
final_model.save_pretrained(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")
print(f"Saved final model + tokenizer to {OUTPUT_DIR}/final_model")

Map:   0%|          | 0/17769 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/apar_xlmrlarge_tapt/tapt_checkpoint
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,2.392246
100,2.185707
150,1.600700
200,0.843993
250,0.712187
300,0.705283
350,0.707764
400,0.549503
450,0.592508
500,0.539956


Parameter count: 559,893,507 (560M)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model + tokenizer to /content/drive/MyDrive/apar_xlmrlarge_tapt/final_model
